# ⚙️ 02 — Preprocesamiento

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

Mounted at /content/drive
✅ Drive montado


In [2]:
%%capture
!pip install lightgbm==4.6.0 optuna scikit-learn pandas numpy mlflow
!pip install fastapi uvicorn nest-asyncio pyngrok evidently joblib matplotlib seaborn
print('OK')

In [3]:
import json, os, glob
import pandas as pd
import numpy as np
import mlflow
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datetime import datetime

BASE_DRIVE = '/content/drive/MyDrive/1. MAESTRÍA CIENCIA DE DATOS/ENTRENAMIENTO PRÁCTICO'

config_path = f'{BASE_DRIVE}/pipeline_config.json'
if os.path.exists(config_path):
    with open(config_path) as f:
        CONFIG = json.load(f)
    print('✅ Config cargada')
else:
    CONFIG = {
        'dir_raw_data'     : f'{BASE_DRIVE}/Data Cruda',
        'dir_training_data': f'{BASE_DRIVE}/Data Entrenamiento',
        'dir_processed'    : f'{BASE_DRIVE}/Data Preprocesada',
        'train_path'       : f'{BASE_DRIVE}/Data Preprocesada/training_data/preprocessed/train_vars_extrac.csv',
        'test_path'        : f'{BASE_DRIVE}/Data Preprocesada/training_data/preprocessed/test_vars_extrac.csv',
        'model_dir'        : f'{BASE_DRIVE}/Modelo',
        'model_path'       : f'{BASE_DRIVE}/Modelo/lgbm_model.pkl',
        'metadata_path'    : f'{BASE_DRIVE}/Modelo/lgbm_metadata.json',
        'mlflow_uri'       : f'{BASE_DRIVE}/mlruns',
        'experiment_name'  : 'campana_credito_propension',
        'monitor_dir'      : f'{BASE_DRIVE}/monitoreo',
        'target_col'       : 'target',
        'id_cols'          : ['partition','key_value','codunicocli',
                              'fch_creacion','p_fecinformacion','seg_un'],
        'score_col'        : 'prob_value_contact',
    }
    print('⚠️ Config definida localmente')

os.makedirs(CONFIG['model_dir'], exist_ok=True)
os.makedirs(os.path.join(CONFIG['model_dir'], 'artifacts'), exist_ok=True)
os.makedirs(os.path.join(CONFIG['dir_processed'], 'training_data', 'preprocessed'), exist_ok=True)

mlflow.set_tracking_uri(CONFIG['mlflow_uri'])
mlflow.set_experiment(CONFIG['experiment_name'])
TARGET_COL = CONFIG['target_col']
print(f'✅ MLflow listo')

✅ Config cargada


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


✅ MLflow listo


## 1. Verificación - datos preprocesados

In [4]:
SKIP = (os.path.exists(CONFIG['train_path']) and
        os.path.exists(CONFIG['test_path']))

if SKIP:
    print('⚡ Datos ya preprocesados — cargando directamente...')
    train_df = pd.read_csv(CONFIG['train_path'])
    test_df  = pd.read_csv(CONFIG['test_path'])
    X_train  = train_df.drop(columns=[TARGET_COL])
    y_train  = train_df[TARGET_COL]
    X_test   = test_df.drop(columns=[TARGET_COL])
    y_test   = test_df[TARGET_COL]
    encoders    = {}
    fill_values = {}
    enc_p = os.path.join(CONFIG['model_dir'], 'artifacts', 'label_encoders.pkl')
    fv_p  = os.path.join(CONFIG['model_dir'], 'artifacts', 'fill_values.pkl')
    if os.path.exists(enc_p): encoders    = joblib.load(enc_p)
    if os.path.exists(fv_p):  fill_values = joblib.load(fv_p)
    print(f'✅ Train: {X_train.shape} | Test: {X_test.shape}')
else:
    print('🔄 Preprocesando desde cero...')

⚡ Datos ya preprocesados — cargando directamente...
✅ Train: (1059077, 60) | Test: (521636, 60)


## 2. Preprocesamiento completo

In [6]:
if not SKIP:
    # Buscar CSVs de entrenamiento
    train_files = sorted(glob.glob(CONFIG['dir_training_data'] + '/**/*.csv', recursive=True))
    if not train_files:
        # Fallback: usar particiones crudas p1-p5
        train_files = sorted(glob.glob(CONFIG['dir_raw_data'] + '/*.csv'))[:5]
        print(f'ℹ️  Usando {len(train_files)} particiones crudas como training')
    else:
        print(f'📂 Archivos de entrenamiento: {len(train_files)}')

    dfs = [pd.read_csv(f) for f in train_files]
    df  = pd.concat(dfs, ignore_index=True)
    print(f'   Cargado: {df.shape[0]:,} × {df.shape[1]}')

    # Eliminar columnas de ID
    drop_cols = [c for c in CONFIG['id_cols'] if c in df.columns]
    df_model  = df.drop(columns=drop_cols)

    y = df_model[TARGET_COL].astype(int)
    X = df_model.drop(columns=[TARGET_COL])

    # Label encoding
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        X[col] = X[col].fillna('MISSING').astype(str)
        X[col] = le.fit_transform(X[col])
        encoders[col] = le

    # Imputación numérica
    fill_values = {}
    for col in X.select_dtypes(include=np.number).columns:
        if X[col].isnull().any():
            med = X[col].median()
            X[col] = X[col].fillna(med)
            fill_values[col] = med

    # Split estratificado
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    # Guardar datasets
    pd.concat([X_train, y_train], axis=1).to_csv(CONFIG['train_path'], index=False)
    pd.concat([X_test,  y_test],  axis=1).to_csv(CONFIG['test_path'],  index=False)

    # Guardar artefactos de preprocesamiento
    art_dir = os.path.join(CONFIG['model_dir'], 'artifacts')
    joblib.dump(encoders,    os.path.join(art_dir, 'label_encoders.pkl'))
    joblib.dump(fill_values, os.path.join(art_dir, 'fill_values.pkl'))

    print(f'\n✅ Preprocesamiento completo')
    print(f'   Train: {X_train.shape} | Target rate: {y_train.mean():.4f}')
    print(f'   Test:  {X_test.shape}  | Target rate: {y_test.mean():.4f}')
    print(f'   Guardado en: {CONFIG["train_path"]}')

In [7]:
# Registrar en MLflow
with mlflow.start_run(run_name='02_preprocesamiento') as run:
    mlflow.log_param('skip_preprocess', SKIP)
    mlflow.log_param('n_features',      X_train.shape[1])
    mlflow.log_param('timestamp',       datetime.now().strftime('%Y-%m-%d %H:%M'))
    mlflow.log_metric('n_train',         len(X_train))
    mlflow.log_metric('n_test',          len(X_test))
    mlflow.log_metric('target_rate_train', float(y_train.mean()))
    mlflow.log_metric('target_rate_test',  float(y_test.mean()))
    print(f'✅ Preprocesamiento registrado → Run ID: {run.info.run_id}')

print('   ➡️  Siguiente: 03_entrenamiento_mlflow.ipynb')

✅ Preprocesamiento registrado → Run ID: 9f1d5bd6e4e04ed5bb0bc2440f3b2b3a
   ➡️  Siguiente: 03_entrenamiento_mlflow.ipynb
